# Healthcare Workforce Scheduling with cuOpt Python API

This notebook demonstrates how to solve a healthcare workforce scheduling problem using the NVIDIA cuOpt Python API. The problem involves assigning nurses and CNAs to hospital shifts while meeting role coverage, charge nurse, PTO, preference, and overtime requirements.

## Problem Description

A hospital ICU unit needs to determine:

- Which registered nurses (RNs) and certified nursing assistants (CNAs) should work each shift
- How to satisfy RN, CNA, and charge nurse demand
- How to respect approved PTO
- How to avoid assigning a staff member to more than one shift per day
- How to minimize uncovered demand, overtime, and preference misses

The goal is to create a feasible staffing plan while prioritizing patient coverage.

### Model Assumptions

**Important**: This model makes the following simplifying assumptions:

1. **Single unit**: The example schedules one ICU unit.
2. **Short planning horizon**: The sample data covers three days with day and night shifts.
3. **Known demand**: Required RN, CNA, and charge nurse counts are fixed for each shift.
4. **Synthetic staff data**: Staff names, roles, preferences, and PTO records are fictional.
5. **Qualified role coverage**: RNs cover RN demand; CNAs cover CNA demand; charge nurse demand requires a charge-eligible RN.
6. **Soft shortfall handling**: Uncovered demand is allowed through slack variables, but heavily penalized.

This formulation is well-suited for:

- Nurse and CNA shift assignment
- ICU coverage planning
- PTO-aware schedule generation
- Charge nurse coverage checks
- Call-out recovery and re-optimization

**Note**: Production nurse rostering may require additional rules such as rest periods, rolling weekend limits, multi-week history, skill tiers, float pools, union rules, and fairness policies.

This problem is formulated as a Mixed Integer Linear Program (MILP). The notebook focuses on cuOpt Python API modeling and solving with a compact, self-contained example.


## Environment Setup

First, let's check if we have a GPU available and install necessary dependencies.



In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

## Import Required Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cuopt.linear_programming import DataModel, Solve, SolverSettings


## Problem Data Setup

Load the synthetic staff roster, shift demand, and PTO records.

- `staff.csv`: staff roster, role, charge eligibility, weekly hour limit, and shift preference
- `demand.csv`: shift-level staffing demand
- `pto.csv`: approved PTO dates

Think of each row in `demand.csv` as a shift that must be filled. In the larger application this came from a SQLite scheduling database; here it is reduced to CSV so the example is self-contained.


In [ ]:
# Works when the notebook is opened from this folder or from the repository root.
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("cuopt_examples/healthcare_workforce_scheduling/data")

staff = pd.read_csv(DATA_DIR / "staff.csv")
demand = pd.read_csv(DATA_DIR / "demand.csv")
pto = pd.read_csv(DATA_DIR / "pto.csv")

demand = demand.reset_index(drop=True)
demand["slot_id"] = [f"slot_{i}" for i in demand.index]

print("Staff")
display(staff)
print("Demand")
display(demand)
print("PTO")
display(pto)


## Problem Formulation

### Decision Variables

- `x[staff, shift]`: 1 if a staff member works a shift, otherwise 0
- `slack[shift, role]`: uncovered staffing demand for a role on a shift
- `overtime[staff]`: hours beyond a staff member's weekly hour limit

### Objective Function

Minimize total scheduling penalty:

```text
uncovered demand penalty + overtime penalty + preference miss penalty
```

### Constraints

1. **RN coverage**: Assigned RNs plus RN slack must meet RN demand.
2. **CNA coverage**: Assigned CNAs plus CNA slack must meet CNA demand.
3. **Charge nurse coverage**: Each shift must have a charge-eligible RN or charge slack.
4. **PTO**: Staff on approved PTO are not eligible for assignment on that date.
5. **One shift per day**: A staff member can work at most one shift per day.
6. **Weekly hours**: Assigned hours should stay within each staff member's limit; overtime is allowed but penalized.


## Define Decision Variables

The main decision variable is:

```text
x[staff, shift] = 1 if the staff member works the shift, else 0
```

For example:

```text
x[RN001, slot_0] = 1  means RN001 works slot_0
x[RN001, slot_0] = 0  means RN001 does not work slot_0
```

Slack variables make the model robust: if a schedule cannot fully cover demand, cuOpt can still return a solution and the uncovered demand is visible in the output.


In [ ]:
var_names = []
lower_bounds = []
upper_bounds = []
integer_flags = []
objective = []

rows = []  # Each row is (coefficients_by_variable_index, sense, rhs)

assignment_vars = {}
slack_vars = {}
overtime_vars = {}

UNMET_DEMAND_PENALTY = 1000.0
OVERTIME_PENALTY = 10.0
PREFERENCE_MISS_PENALTY = 2.0


def add_var(name, lb=0.0, ub=1.0, integer=True, cost=0.0):
    idx = len(var_names)
    var_names.append(name)
    lower_bounds.append(float(lb))
    upper_bounds.append(float(ub))
    integer_flags.append(bool(integer))
    objective.append(float(cost))
    return idx


def add_constraint(coeffs, sense, rhs):
    # sense: "G" means >=, "L" means <=, "E" means equality
    rows.append((dict(coeffs), sense, float(rhs)))


def is_day_shift(shift):
    return shift.startswith("D")


pto_pairs = set(zip(pto["staff_id"], pto["date"]))

for _, slot in demand.iterrows():
    for _, person in staff.iterrows():
        sid = person["staff_id"]
        role = person["role"]

        # PTO means no assignment variable is created for that person/date.
        if (sid, slot["date"]) in pto_pairs:
            continue

        # RNs cover RN demand. CNAs cover CNA demand.
        if role == "RN" and slot["required_rn"] <= 0:
            continue
        if role == "CNA" and slot["required_cna"] <= 0:
            continue

        preference = person["shift_preference"]
        preference_cost = 0.0
        if preference == "day" and not is_day_shift(slot["shift"]):
            preference_cost = PREFERENCE_MISS_PENALTY
        elif preference == "night" and is_day_shift(slot["shift"]):
            preference_cost = PREFERENCE_MISS_PENALTY

        key = (sid, slot["slot_id"])
        assignment_vars[key] = add_var(
            ("x", sid, slot["slot_id"]),
            lb=0,
            ub=1,
            integer=True,
            cost=preference_cost,
        )

print("Assignment variables:", len(assignment_vars))
print("Example variables:", var_names[:5])

## Set Objective Function

The objective is encoded when variables are created:

- `UNMET_DEMAND_PENALTY = 1000`: leaving a shift short-staffed is very expensive.
- `OVERTIME_PENALTY = 10`: overtime is allowed but discouraged.
- `PREFERENCE_MISS_PENALTY = 2`: preference misses are allowed when needed, but lightly penalized.

This priority order tells cuOpt to protect patient coverage first, then improve schedule quality where possible.


## Add Constraints

Add coverage, charge nurse, PTO, one-shift-per-day, and weekly-hour constraints.

For every shift, the model requires enough RNs and CNAs. Example:

```text
sum(RN assignments for this shift) + RN slack >= required RN count
```

For charge nurse coverage, at least one charge-eligible RN is required on each shift.


In [ ]:
for _, slot in demand.iterrows():
    slot_id = slot["slot_id"]

    # RN coverage
    rn_vars = [
        vi
        for (sid, sid_slot), vi in assignment_vars.items()
        if sid_slot == slot_id and staff.loc[staff.staff_id == sid, "role"].iloc[0] == "RN"
    ]
    rn_slack = add_var(("slack", slot_id, "RN"), lb=0, ub=slot["required_rn"], integer=True, cost=UNMET_DEMAND_PENALTY)
    slack_vars[(slot_id, "RN")] = rn_slack
    add_constraint({**{vi: 1.0 for vi in rn_vars}, rn_slack: 1.0}, "G", slot["required_rn"])

    # CNA coverage
    cna_vars = [
        vi
        for (sid, sid_slot), vi in assignment_vars.items()
        if sid_slot == slot_id and staff.loc[staff.staff_id == sid, "role"].iloc[0] == "CNA"
    ]
    cna_slack = add_var(("slack", slot_id, "CNA"), lb=0, ub=slot["required_cna"], integer=True, cost=UNMET_DEMAND_PENALTY)
    slack_vars[(slot_id, "CNA")] = cna_slack
    add_constraint({**{vi: 1.0 for vi in cna_vars}, cna_slack: 1.0}, "G", slot["required_cna"])

    # Charge nurse coverage: charge-eligible RNs only.
    charge_vars = []
    for (sid, sid_slot), vi in assignment_vars.items():
        if sid_slot != slot_id:
            continue
        person = staff.loc[staff.staff_id == sid].iloc[0]
        if person["role"] == "RN" and int(person["charge_eligible"]) == 1:
            charge_vars.append(vi)

    charge_slack = add_var(("slack", slot_id, "CHARGE"), lb=0, ub=slot["required_charge"], integer=True, cost=UNMET_DEMAND_PENALTY)
    slack_vars[(slot_id, "CHARGE")] = charge_slack
    add_constraint({**{vi: 1.0 for vi in charge_vars}, charge_slack: 1.0}, "G", slot["required_charge"])

print("Constraints after coverage:", len(rows))

### Add Staff Rules

Coverage alone would only answer: "Can every shift be filled?"

A useful workforce schedule also needs staff-level rules:

1. A staff member can work at most one shift per day.
2. A staff member should stay within weekly hours. Overtime is allowed but penalized.

These rules are what make the problem an optimization problem instead of a simple matching table.


In [ ]:
# At most one shift per person per day.
for sid in staff["staff_id"]:
    for shift_date in demand["date"].unique():
        vars_for_day = []
        slots_for_day = demand.loc[demand["date"] == shift_date, "slot_id"]
        for slot_id in slots_for_day:
            vi = assignment_vars.get((sid, slot_id))
            if vi is not None:
                vars_for_day.append(vi)
        if vars_for_day:
            add_constraint({vi: 1.0 for vi in vars_for_day}, "L", 1)

# Weekly hour limit with overtime variable.
for _, person in staff.iterrows():
    sid = person["staff_id"]
    staff_assignment_terms = {}
    for _, slot in demand.iterrows():
        vi = assignment_vars.get((sid, slot["slot_id"]))
        if vi is not None:
            staff_assignment_terms[vi] = float(slot["hours"])

    overtime = add_var(("overtime", sid), lb=0, ub=48, integer=False, cost=OVERTIME_PENALTY)
    overtime_vars[sid] = overtime

    # total assigned hours - overtime <= max_hours
    add_constraint({**staff_assignment_terms, overtime: -1.0}, "L", float(person["max_hours"]))

print("Total variables:", len(var_names))
print("Total constraints:", len(rows))

## Build the cuOpt MILP Model

cuOpt receives the model in array form:

- CSR constraint matrix
- lower and upper constraint bounds
- variable lower and upper bounds
- variable types
- objective coefficients

The model minimizes the objective, so lower objective value is better.


In [ ]:
INF = float("inf")

offsets = [0]
indices = []
values = []
constraint_lowers = []
constraint_uppers = []

for coeffs, sense, rhs in rows:
    for variable_index, coeff in coeffs.items():
        indices.append(variable_index)
        values.append(float(coeff))
    offsets.append(len(indices))

    if sense == "G":
        constraint_lowers.append(rhs)
        constraint_uppers.append(INF)
    elif sense == "L":
        constraint_lowers.append(-INF)
        constraint_uppers.append(rhs)
    elif sense == "E":
        constraint_lowers.append(rhs)
        constraint_uppers.append(rhs)
    else:
        raise ValueError(f"Unknown constraint sense: {sense}")

model = DataModel()
model.set_csr_constraint_matrix(
    np.array(values, dtype=np.float64),
    np.array(indices, dtype=np.int32),
    np.array(offsets, dtype=np.int32),
)
model.set_constraint_lower_bounds(np.array(constraint_lowers, dtype=np.float64))
model.set_constraint_upper_bounds(np.array(constraint_uppers, dtype=np.float64))
model.set_variable_lower_bounds(np.array(lower_bounds, dtype=np.float64))
model.set_variable_upper_bounds(np.array(upper_bounds, dtype=np.float64))
model.set_variable_types(np.array(["I" if flag else "C" for flag in integer_flags]))
model.set_objective_coefficients(np.array(objective, dtype=np.float64))

settings = SolverSettings()
settings.set_parameter("time_limit", 60)

print("cuOpt model ready")
print("Variables:", len(var_names))
print("Constraints:", len(rows))

## Solve the Problem

The next cell calls the cuOpt Python API directly to solve the MILP model.

The important lines to look for in the solver output are:

- the GPU/device being used
- the number of variables and constraints
- `Termination: Optimal` or another solve status
- the final objective value


In [ ]:
solution = Solve(model, settings)
primal = np.array(solution.get_primal_solution(), dtype=np.float64)
termination = str(solution.get_termination_reason()).split(".")[-1]
objective_value = float(solution.get_primal_objective())

if len(primal) != len(var_names):
    raise RuntimeError(f"cuOpt returned {len(primal)} values for {len(var_names)} variables")

print("Termination:", termination)
print("Objective value:", objective_value)

## Analyze Results

Convert variables with value `1` back into staff-to-shift assignments and report uncovered demand from slack variables.


In [ ]:
def solution_value(index):
    value = float(primal[index])
    if integer_flags[index]:
        return round(value)
    return value

assignments = []
open_shifts = []
overtime_rows = []

for index, name in enumerate(var_names):
    value = solution_value(index)
    if value <= 1e-6:
        continue

    if name[0] == "x":
        _, sid, slot_id = name
        slot = demand.loc[demand.slot_id == slot_id].iloc[0]
        person = staff.loc[staff.staff_id == sid].iloc[0]
        assignments.append({
            "date": slot["date"],
            "unit": slot["unit"],
            "shift": slot["shift"],
            "staff_id": sid,
            "name": person["name"],
            "role": person["role"],
            "charge_eligible": int(person["charge_eligible"]),
        })
    elif name[0] == "slack":
        _, slot_id, role = name
        slot = demand.loc[demand.slot_id == slot_id].iloc[0]
        open_shifts.append({
            "date": slot["date"],
            "unit": slot["unit"],
            "shift": slot["shift"],
            "role": role,
            "missing": value,
        })
    elif name[0] == "overtime":
        _, sid = name
        overtime_rows.append({"staff_id": sid, "overtime_hours": round(value, 2)})

schedule = pd.DataFrame(assignments).sort_values(["date", "shift", "role", "staff_id"])
open_demand = pd.DataFrame(open_shifts)
overtime = pd.DataFrame(overtime_rows)

print("Assignments:", len(schedule))
print("Open demand rows:", len(open_demand))
print("Objective value:", objective_value)

display(schedule)
if len(open_demand):
    display(open_demand)
if len(overtime):
    display(overtime)

## Visualize Results

The schedule table is useful for exact assignments, but a simple heatmap makes the staffing pattern easier to scan.

The grid below shows each staff member by date:

- **Day**: assigned to the 07:00-19:00 shift
- **Night**: assigned to the 19:00-07:00 shift
- **Off**: not scheduled that day
- **PTO**: approved time off, so the staff member is unavailable

The coverage table below the chart summarizes how many RNs and CNAs were assigned to each shift. In this example, every shift meets its required coverage.


In [ ]:
def plot_schedule_grid(schedule_df, title):
    dates = [str(d) for d in demand["date"].unique()]
    staff_order = [str(sid) for sid in staff["staff_id"]]
    staff_labels = dict(zip(staff["staff_id"].astype(str), staff["name"]))

    grid = pd.DataFrame("Off", index=staff_order, columns=dates)

    # Mark approved PTO separately from normal off days.
    for _, row in pto.iterrows():
        staff_id = str(row["staff_id"])
        pto_date = str(row["date"])
        if staff_id in grid.index and pto_date in grid.columns:
            grid.loc[staff_id, pto_date] = "PTO"

    # Fill solved assignments.
    for _, row in schedule_df.iterrows():
        staff_id = str(row["staff_id"])
        shift_date = str(row["date"])
        shift_label = "Day" if str(row["shift"]).startswith("D") else "Night"

        if staff_id in grid.index and shift_date in grid.columns:
            grid.loc[staff_id, shift_date] = shift_label

    color_map = {"Off": 0, "PTO": 1, "Day": 2, "Night": 3}
    numeric_grid = grid.replace(color_map).astype(int)

    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.imshow(
        numeric_grid.to_numpy(),
        aspect="auto",
        cmap=plt.cm.Set2,
        vmin=0,
        vmax=3,
    )

    ax.set_xticks(range(len(dates)))
    ax.set_xticklabels(dates, rotation=30, ha="right")
    ax.set_yticks(range(len(staff_order)))
    ax.set_yticklabels([f"{sid} - {staff_labels[sid]}" for sid in staff_order])
    ax.set_title(title)

    for y in range(len(staff_order)):
        for x in range(len(dates)):
            ax.text(x, y, grid.iloc[y, x], ha="center", va="center", fontsize=9)

    ax.set_xlabel("Date")
    ax.set_ylabel("Staff")
    plt.tight_layout()
    plt.show()


plot_schedule_grid(schedule, "Optimized ICU Schedule")

coverage_summary = (
    schedule.groupby(["date", "shift", "role"])
    .size()
    .reset_index(name="assigned")
    .sort_values(["date", "shift", "role"])
)

print("Coverage summary by shift and role")
display(coverage_summary)

## Summary and Business Insights

A useful schedule should have:

- solver termination indicating a solution was found
- low objective value
- zero or minimal uncovered demand
- readable assignments that can be handed back to an operational scheduling system

For this tiny dataset, an objective of `0.0` means cuOpt found a schedule with no uncovered demand, no overtime penalty, and no preference penalty.


In [ ]:
summary = {
    "termination": termination,
    "objective_value": objective_value,
    "assignments": len(schedule),
    "open_shift_rows": len(open_demand),
    "open_staff_count": int(open_demand["missing"].sum()) if len(open_demand) else 0,
}
summary

## Call-Out Re-optimization

A common hospital scheduling workflow is a last-minute call-out: someone who was assigned to a shift can no longer work.

In this section, we simulate a slightly harder recovery case:

1. A scheduled night RN calls out.
2. The same night needs one extra RN because demand shifts toward the night shift.
3. cuOpt rebuilds and solves the MILP again.

The model treats coverage as more important than shift preference:

```text
uncovered demand penalty = 1000
preference miss penalty = 2
```

So if all preferred options are unavailable, cuOpt may assign a qualified staff member to a less-preferred shift. That keeps patient coverage complete, but adds a small preference penalty to the objective.

This is not brute force. Brute force would try every possible schedule one by one. cuOpt uses mathematical optimization methods, presolve, bounds, and search to find the best solution for the MILP and prove optimality when possible.

The expected result is therefore not "perfect again." Instead, the solver should keep all shifts covered while accepting a small preference penalty if it has to move a day-preference nurse onto a night shift.

This is the same optimization pattern used by larger scheduling systems:

```text
current schedule + disruption -> rebuild constraints -> solve again -> updated schedule
```


In [ ]:
def solve_staffing_problem(extra_unavailable=None, label="base", demand_frame=None):
    extra_unavailable = set(extra_unavailable or [])
    unavailable_pairs = set(zip(pto["staff_id"], pto["date"])) | extra_unavailable
    active_demand = (demand if demand_frame is None else demand_frame).copy()

    local_var_names = []
    local_lower_bounds = []
    local_upper_bounds = []
    local_integer_flags = []
    local_objective = []
    local_rows = []
    local_assignment_vars = {}

    def local_add_var(name, lb=0.0, ub=1.0, integer=True, cost=0.0):
        idx = len(local_var_names)
        local_var_names.append(name)
        local_lower_bounds.append(float(lb))
        local_upper_bounds.append(float(ub))
        local_integer_flags.append(bool(integer))
        local_objective.append(float(cost))
        return idx

    def local_add_constraint(coeffs, sense, rhs):
        local_rows.append((dict(coeffs), sense, float(rhs)))

    for _, slot in active_demand.iterrows():
        for _, person in staff.iterrows():
            sid = person["staff_id"]
            role = person["role"]

            if (sid, slot["date"]) in unavailable_pairs:
                continue
            if role == "RN" and slot["required_rn"] <= 0:
                continue
            if role == "CNA" and slot["required_cna"] <= 0:
                continue

            preference = person["shift_preference"]
            preference_cost = 0.0
            if preference == "day" and not is_day_shift(slot["shift"]):
                preference_cost = PREFERENCE_MISS_PENALTY
            elif preference == "night" and is_day_shift(slot["shift"]):
                preference_cost = PREFERENCE_MISS_PENALTY

            local_assignment_vars[(sid, slot["slot_id"])] = local_add_var(
                ("x", sid, slot["slot_id"]),
                lb=0,
                ub=1,
                integer=True,
                cost=preference_cost,
            )

    for _, slot in active_demand.iterrows():
        slot_id = slot["slot_id"]

        rn_vars = [
            vi
            for (sid, sid_slot), vi in local_assignment_vars.items()
            if sid_slot == slot_id and staff.loc[staff.staff_id == sid, "role"].iloc[0] == "RN"
        ]
        rn_slack = local_add_var(("slack", slot_id, "RN"), lb=0, ub=slot["required_rn"], integer=True, cost=UNMET_DEMAND_PENALTY)
        local_add_constraint({**{vi: 1.0 for vi in rn_vars}, rn_slack: 1.0}, "G", slot["required_rn"])

        cna_vars = [
            vi
            for (sid, sid_slot), vi in local_assignment_vars.items()
            if sid_slot == slot_id and staff.loc[staff.staff_id == sid, "role"].iloc[0] == "CNA"
        ]
        cna_slack = local_add_var(("slack", slot_id, "CNA"), lb=0, ub=slot["required_cna"], integer=True, cost=UNMET_DEMAND_PENALTY)
        local_add_constraint({**{vi: 1.0 for vi in cna_vars}, cna_slack: 1.0}, "G", slot["required_cna"])

        charge_vars = []
        for (sid, sid_slot), vi in local_assignment_vars.items():
            if sid_slot != slot_id:
                continue
            person = staff.loc[staff.staff_id == sid].iloc[0]
            if person["role"] == "RN" and int(person["charge_eligible"]) == 1:
                charge_vars.append(vi)

        charge_slack = local_add_var(("slack", slot_id, "CHARGE"), lb=0, ub=slot["required_charge"], integer=True, cost=UNMET_DEMAND_PENALTY)
        local_add_constraint({**{vi: 1.0 for vi in charge_vars}, charge_slack: 1.0}, "G", slot["required_charge"])

    for sid in staff["staff_id"]:
        for shift_date in active_demand["date"].unique():
            vars_for_day = []
            slots_for_day = active_demand.loc[active_demand["date"] == shift_date, "slot_id"]
            for slot_id in slots_for_day:
                vi = local_assignment_vars.get((sid, slot_id))
                if vi is not None:
                    vars_for_day.append(vi)
            if vars_for_day:
                local_add_constraint({vi: 1.0 for vi in vars_for_day}, "L", 1)

    for _, person in staff.iterrows():
        sid = person["staff_id"]
        terms = {}
        for _, slot in active_demand.iterrows():
            vi = local_assignment_vars.get((sid, slot["slot_id"]))
            if vi is not None:
                terms[vi] = float(slot["hours"])
        overtime_idx = local_add_var(("overtime", sid), lb=0, ub=48, integer=False, cost=OVERTIME_PENALTY)
        local_add_constraint({**terms, overtime_idx: -1.0}, "L", float(person["max_hours"]))

    local_offsets = [0]
    local_indices = []
    local_values = []
    local_constraint_lowers = []
    local_constraint_uppers = []

    for coeffs, sense, rhs in local_rows:
        for variable_index, coeff in coeffs.items():
            local_indices.append(variable_index)
            local_values.append(float(coeff))
        local_offsets.append(len(local_indices))

        if sense == "G":
            local_constraint_lowers.append(rhs)
            local_constraint_uppers.append(INF)
        elif sense == "L":
            local_constraint_lowers.append(-INF)
            local_constraint_uppers.append(rhs)
        elif sense == "E":
            local_constraint_lowers.append(rhs)
            local_constraint_uppers.append(rhs)
        else:
            raise ValueError(f"Unknown constraint sense: {sense}")

    local_model = DataModel()
    local_model.set_csr_constraint_matrix(
        np.array(local_values, dtype=np.float64),
        np.array(local_indices, dtype=np.int32),
        np.array(local_offsets, dtype=np.int32),
    )
    local_model.set_constraint_lower_bounds(np.array(local_constraint_lowers, dtype=np.float64))
    local_model.set_constraint_upper_bounds(np.array(local_constraint_uppers, dtype=np.float64))
    local_model.set_variable_lower_bounds(np.array(local_lower_bounds, dtype=np.float64))
    local_model.set_variable_upper_bounds(np.array(local_upper_bounds, dtype=np.float64))
    local_model.set_variable_types(np.array(["I" if flag else "C" for flag in local_integer_flags]))
    local_model.set_objective_coefficients(np.array(local_objective, dtype=np.float64))

    local_solution = Solve(local_model, settings)
    local_primal = np.array(local_solution.get_primal_solution(), dtype=np.float64)

    local_assignments = []
    local_open_shifts = []
    for index, name in enumerate(local_var_names):
        value = round(float(local_primal[index])) if local_integer_flags[index] else float(local_primal[index])
        if value <= 1e-6:
            continue

        if name[0] == "x":
            _, sid, slot_id = name
            slot = active_demand.loc[active_demand.slot_id == slot_id].iloc[0]
            person = staff.loc[staff.staff_id == sid].iloc[0]
            local_assignments.append({
                "date": slot["date"],
                "unit": slot["unit"],
                "shift": slot["shift"],
                "staff_id": sid,
                "name": person["name"],
                "role": person["role"],
                "charge_eligible": int(person["charge_eligible"]),
            })
        elif name[0] == "slack":
            _, slot_id, role = name
            slot = active_demand.loc[active_demand.slot_id == slot_id].iloc[0]
            local_open_shifts.append({
                "date": slot["date"],
                "unit": slot["unit"],
                "shift": slot["shift"],
                "role": role,
                "missing": value,
            })

    return {
        "label": label,
        "termination": str(local_solution.get_termination_reason()).split(".")[-1],
        "objective_value": float(local_solution.get_primal_objective()),
        "schedule": pd.DataFrame(local_assignments).sort_values(["date", "shift", "role", "staff_id"]),
        "open_demand": pd.DataFrame(local_open_shifts),
        "variables": len(local_var_names),
        "constraints": len(local_rows),
    }


In [ ]:
called_out_staff = "RN002"
called_out_date = "2026-08-03"
called_out_name = staff.loc[staff.staff_id == called_out_staff, "name"].iloc[0]

scenario_demand = demand.copy()
scenario_demand.loc[
    (scenario_demand["date"] == called_out_date) & (scenario_demand["shift"] == "D07-19"),
    "required_rn",
] = 1
scenario_demand.loc[
    (scenario_demand["date"] == called_out_date) & (scenario_demand["shift"] == "N19-07"),
    "required_rn",
] = 3

print(f"Call-out scenario: {called_out_name} ({called_out_staff}) is unavailable on {called_out_date}.")
print("Stress case: RN demand shifts from the day shift to the night shift on the same date.")
print("Before call-out, this staff member was assigned:")
display(schedule[(schedule.staff_id == called_out_staff) & (schedule.date == called_out_date)])

print("Scenario demand for affected date:")
display(scenario_demand[scenario_demand.date == called_out_date])

callout_result = solve_staffing_problem(
    extra_unavailable={(called_out_staff, called_out_date)},
    label="callout",
    demand_frame=scenario_demand,
)

callout_schedule = callout_result["schedule"]
callout_open_demand = callout_result["open_demand"]

print("After call-out re-optimization")
print("Termination:", callout_result["termination"])
print("Objective value:", callout_result["objective_value"])
print("Assignments:", len(callout_schedule))
print("Open demand rows:", len(callout_open_demand))

print("Updated assignments for the affected date:")
display(callout_schedule[callout_schedule.date == called_out_date])

if callout_result["objective_value"] > 0 and len(callout_open_demand) == 0:
    print("Tradeoff shown: all demand is covered, but cuOpt accepted a small scheduling penalty.")

if len(callout_open_demand):
    print("Open demand after call-out:")
    display(callout_open_demand)


In this call-out scenario, cuOpt prioritized patient coverage first. It kept all shifts covered and accepted a small penalty of 2.0 because one replacement assignment did not match the staff member's shift preference.

### Visualize Call-Out Impact

The call-out recovery can also be compared visually. The base schedule is the normal plan; the call-out schedule is the re-optimized plan after demand shifts and the assigned night RN becomes unavailable.


In [ ]:
plot_schedule_grid(callout_schedule, "Re-optimized Schedule After Call-Out")

affected_date = "2026-08-03"
affected_shift = "N19-07"
called_out_staff = "RN002"

before = schedule[
    (schedule["date"].astype(str) == affected_date)
    & (schedule["shift"] == affected_shift)
].sort_values("staff_id")

after = callout_schedule[
    (callout_schedule["date"].astype(str) == affected_date)
    & (callout_schedule["shift"] == affected_shift)
].sort_values("staff_id")

print("Affected shift story")
print(f"- Date: {affected_date}")
print(f"- Shift: {affected_shift}")
print("- Brooke Lee (RN002) called out and became unavailable.")
print("- Scenario demand increased night RN need from 2 to 3.")
print("- cuOpt kept open demand at 0 by re-optimizing the assignments.")
print("- Objective increased to 2.0 because one assignment does not match a shift preference.")

print("\nBefore call-out, the affected night shift had:")
display(before[["staff_id", "name", "role", "shift"]])

print("\nAfter call-out, the affected night shift has:")
display(after[["staff_id", "name", "role", "shift"]])

removed = sorted(set(before["staff_id"]) - set(after["staff_id"]))
added = sorted(set(after["staff_id"]) - set(before["staff_id"]))

print("\nAssignment change:")
print("Removed:", removed if removed else "None")
print("Added:", added if added else "None")

## Conclusion

This notebook demonstrated how to use the cuOpt Python API to solve a healthcare workforce scheduling problem as a MILP.

### What We Achieved

1. **Staff-to-shift assignments** for nurses and CNAs
2. **Coverage enforcement** for RN, CNA, and charge nurse requirements
3. **PTO-aware scheduling** by excluding unavailable staff from assignment variables
4. **Penalty-based tradeoffs** for uncovered demand, overtime, and preference misses
5. **Call-out re-optimization** by rebuilding and resolving the MILP after a disruption

### Key Advantages of cuOpt

- **Optimization modeling**: Express staffing rules as variables, constraints, and an objective.
- **GPU-aware solving**: Uses cuOpt's accelerated optimization engine where applicable.
- **Operational recovery**: Re-solves when call-outs or demand changes occur.
- **Python integration**: Fits naturally into notebook and application workflows.

### Current Model Assumptions

This example is intentionally small and synthetic. It is designed to teach the cuOpt modeling workflow, not to be a full production nurse rostering benchmark.

### Possible Extensions

**High Priority - Larger Healthcare Scheduling Cases:**

- Add more units and staff
- Extend the planning horizon to one or more weeks
- Add rest-time and fatigue constraints
- Add certification and skill-specific requirements
- Add weekend limits and rolling history rules

**Additional Enhancements:**

- Add float pool and agency staff as high-cost fallback options
- Add fairness and workload balancing objectives
- Model multiple acuity levels and census-driven demand
- Connect output to a scheduling dashboard or notification workflow


SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
